# 03 · Feature preparation for the EDA dashboard

This notebook turns the raw ACDC member file into one clean table for the interactive dashboard (`data/processed/acdc_dash.parquet`).

It follows the cleaning decisions from `02_data_understanding.ipynb` and adds a small set of business features. Every value we change is flagged, so the dashboard can show what was corrected and the reader can exclude flagged rows.

What this notebook does not do:
- It does not impute missing values. Remaining gaps stay as missing and are handled in the Deliverable 2 preprocessing pipeline.
- It does not create clusters or segment labels. The member states defined below are simple business rules, not model output.

Run order: execute all cells from top to bottom. The notebook only reads `data/raw/` and only writes to `data/processed/` and `docs/`.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 60)
pd.set_option("display.float_format", "{:,.2f}".format)

# Paths are relative to the repository root, so the notebook runs on any machine.
ROOT = Path.cwd().parent if Path.cwd().name == "notebook" else Path.cwd()
RAW_FILE = ROOT / "data" / "raw" / "acdc_dataset.csv"
OUT_FILE = ROOT / "data" / "processed" / "acdc_dash.parquet"
DICT_FILE = ROOT / "docs" / "dash_data_dictionary.md"

EXTRACTION_DATE = pd.Timestamp("2024-01-01")  # verified in section 2
DORMANT_DAYS = 60                              # active members unseen for longer are "dormant"
NEW_MEMBER_DAYS = 90                           # active members with shorter tenure are "new"

## 1. Load the raw file

The file uses `;` as separator. We keep an untouched copy (`raw`) so every correction can be compared with the original value.

In [2]:
raw = pd.read_csv(RAW_FILE, sep=";")
df = raw.copy()
print(f"Raw shape: {raw.shape[0]:,} rows x {raw.shape[1]} columns")

Raw shape: 15,465 rows x 31 columns


## 2. Dates and the extraction date

Dates come in two formats: ISO (2023-03-06) and day first (06/03/2023). We try ISO first and then day first, so no date is read with day and month swapped.

membership_end = 1970-01-01 is a placeholder for members who are still active. We store this as is_active and leave the end date empty instead of inventing one.

The extraction date is not given in the data. For active members, last_badge_swipe + days_since_last_visit should equal it, so we check that below.

In [3]:
def parse_dates(s: pd.Series) -> pd.Series:
    """Parse ISO dates first, then day-first dates for the rest."""
    iso = pd.to_datetime(s, format="%Y-%m-%d", errors="coerce")
    day_first = pd.to_datetime(s, format="%d/%m/%Y", errors="coerce")
    return iso.fillna(day_first)


for col in ["membership_start", "membership_end", "last_badge_swipe"]:
    df[col] = parse_dates(df[col])
    print(f"{col:<18} unparsed: {df[col].isna().sum()}")

df["is_active"] = df["membership_end"].eq(pd.Timestamp("1970-01-01"))
df.loc[df["is_active"], "membership_end"] = pd.NaT

# For active members, swipe + days since last visit should land on the extraction date.
valid_days = raw["days_since_last_visit"].where(raw["days_since_last_visit"].between(0, 998))
implied = (df["last_badge_swipe"] + pd.to_timedelta(valid_days, unit="D"))[df["is_active"]]
print(f"\nActive members: {df['is_active'].sum():,}")
print(implied.value_counts().head(3))

membership_start   unparsed: 0
membership_end     unparsed: 0
last_badge_swipe   unparsed: 0

Active members: 3,956
2024-01-01    3288
2023-12-31     231
2023-12-28       4
Name: count, dtype: int64


For most active members the implied date is 2024-01-01, so we use it as the extraction date. The few rows with other dates are the swipe errors handled in section 5.

For members who left, days_since_last_visit is counted up to their end date, not up to the extraction date. We use this in section 5 when rebuilding swipe dates.

## 3. Text fields

lifetime_value is stored as text with euro signs, spaces and thousands separators (`4,311.15`). We remove the separators before converting, so large values are not lost.

Category labels are standardised with the same mapping as notebook 02. `unknown` is treated as missing.

In [4]:
ltv_text = (
    raw["lifetime_value"].astype("string")
    .str.replace(r"[€â‚¬\s]", "", regex=True)       # euro sign and its broken encoding
    .str.replace(r",(?=\d{3}(?:\D|$))", "", regex=True)  # thousands separator
    .str.replace(",", ".", regex=False)              # any remaining decimal comma
)
df["lifetime_value"] = pd.to_numeric(ltv_text, errors="coerce")

lost = df["lifetime_value"].isna() & raw["lifetime_value"].notna() & raw["lifetime_value"].astype(str).ne("nan")
print(f"Lifetime values that could not be read: {lost.sum()}")
print(f"Lifetime values missing in the raw file: {raw['lifetime_value'].isna().sum():,}")

Lifetime values that could not be read: 0
Lifetime values missing in the raw file: 1,040


In [5]:
CATEGORY_MAP = {
    "gender": {"f": "female", "m": "male", "o": "other"},
    "industry": {
        "it": "information technology", "tech": "information technology",
        "marketing/pr": "marketing & pr", "marketing and pr": "marketing & pr",
        "design/creative": "design & creative", "design and creative": "design & creative",
        "realestate": "real estate", "real-estate": "real estate",
    },
    "country": {
        "portucal": "portugal", "pt": "portugal",
        "uk": "united kingdom", "gb": "united kingdom", "u.k.": "united kingdom",
        "de": "germany", "deutschland": "germany",
        "es": "spain", "espana": "spain", "españa": "spain",
        "fr": "france", "nl": "netherlands", "holland": "netherlands",
        "the netherlands": "netherlands", "ie": "ireland",
    },
    "plan_tier": {
        "hot desk": "hot_desk", "hotdesk": "hot_desk",
        "full time": "full_time", "fulltime": "full_time",
        "dedicated desk": "dedicated_desk", "dedicateddesk": "dedicated_desk",
        "private office": "private_office", "privateoffice": "private_office",
    },
}

for col, mapping in CATEGORY_MAP.items():
    clean = df[col].astype("string").str.strip().str.lower().replace(mapping)
    df[col] = clean.replace({"unknown": pd.NA})
    print(f"{col:<10} {df[col].nunique()} categories, {df[col].isna().sum()} missing")

df["uses_print"] = df["uses_print"].astype(bool)  # stored as 0/1, the other add-ons as True/False

gender     3 categories, 530 missing
industry   10 categories, 696 missing
country    8 categories, 430 missing
plan_tier  5 categories, 105 missing


## 4. Duplicates

Fifteen rows are exact copies of another row. We drop them and check that `member_id` is then unique.

In [6]:
n_before = len(df)
df = df[~raw.duplicated()].copy()
raw = raw.loc[df.index]  # keep the raw copy aligned with df
print(f"Exact duplicates dropped: {n_before - len(df)}")
print(f"member_id unique after drop: {df['member_id'].is_unique}")

Exact duplicates dropped: 15
member_id unique after drop: True


## 5. Cleaning rules

Each rule below was agreed in the group. When a rule changes a value, a flag_* column records it. The table at the end of this section counts every flag.

| Issue | Rule |
|---|---|
| Age outside 16 to 100 | Set to missing |
| plan_days_per_week of 0 or 9 | Use the plan's standard days (each plan has exactly one valid value) |
| entitled_visits_last_period = 0 | Use the median for the same plan |
| days_since_last_visit = 999 | Set to missing (missing-value code) |
| n_colleagues_on_account = -1 | 0 for individual accounts, missing for team accounts |
| Negative add-on hours or passes | 0 if the add-on is not used, otherwise missing |
| Negative visits, renewals or lifetime value | Set to missing |
| Start date after end date | End = start + total visits / weekly visits (in weeks), otherwise missing |
| Last swipe before start or after the extraction date | Swipe = reference date − days since last visit, otherwise missing |
| Last-period visits above total visits | Cap last-period visits at total visits |
| Monthly value above ten times the plan median | Flag only (the value is kept) |

The reference date for the swipe rule is the end date for members who left and the extraction date for active members (see section 2).

In [7]:
flags = pd.DataFrame(index=df.index)

# Age
flags["age"] = df["age"].notna() & ~df["age"].between(16, 100)
df.loc[flags["age"], "age"] = np.nan

# Plan days: each plan has one standard value, so 0 and 9 are entry errors.
PLAN_DAYS = {"hot_desk": 2, "flex": 3, "dedicated_desk": 5, "full_time": 5, "private_office": 7}
standard_days = df["plan_tier"].map(PLAN_DAYS)
flags["plan_days"] = ~df["plan_days_per_week"].isin(PLAN_DAYS.values())
df.loc[flags["plan_days"], "plan_days_per_week"] = standard_days[flags["plan_days"]]

# Entitled visits of zero: use the plan median.
plan_median = df.loc[df["entitled_visits_last_period"] > 0].groupby("plan_tier")["entitled_visits_last_period"].median()
flags["entitled_visits"] = df["entitled_visits_last_period"].eq(0)
df.loc[flags["entitled_visits"], "entitled_visits_last_period"] = df["plan_tier"].map(plan_median)[flags["entitled_visits"]]

# Missing-value code in days since last visit.
flags["days_code_999"] = df["days_since_last_visit"].eq(999)
df["days_since_last_visit"] = df["days_since_last_visit"].astype("float").mask(flags["days_code_999"])

# Colleagues on account: -1 is a missing-value code.
code_minus1 = df["n_colleagues_on_account"].eq(-1)
flags["colleagues"] = code_minus1
df["n_colleagues_on_account"] = df["n_colleagues_on_account"].astype("float")
df.loc[code_minus1 & ~df["is_team_account"], "n_colleagues_on_account"] = 0
df.loc[code_minus1 & df["is_team_account"], "n_colleagues_on_account"] = np.nan

# Negative add-on quantities.
ADDON_QTY = {
    "meeting_room_hours": "uses_meeting_rooms",
    "event_space_hours": "uses_event_space",
    "guest_passes_issued": "uses_guest_passes",
}
neg_addon = pd.Series(False, index=df.index)
for qty, used in ADDON_QTY.items():
    neg = df[qty] < 0
    neg_addon |= neg
    df.loc[neg, qty] = np.where(df.loc[neg, used], np.nan, 0)
flags["negative_addon"] = neg_addon

# Negative counts and value.
neg_any = pd.Series(False, index=df.index)
for col in ["total_visits", "n_renewals", "lifetime_value"]:
    neg = df[col] < 0
    neg_any |= neg
    df[col] = df[col].astype("float").mask(neg)
flags["negative_value"] = neg_any

In [8]:
start, end, swipe = df["membership_start"], df["membership_end"], df["last_badge_swipe"]

# Start after end: rebuild the end date from total visits and the weekly visit rate.
flags["start_after_end"] = ~df["is_active"] & (start > end)
weeks = df["total_visits"] / df["avg_weekly_visits"].where(df["avg_weekly_visits"] > 0)
rebuilt_end = start + pd.to_timedelta((weeks * 7).round(), unit="D")
rebuilt_end = rebuilt_end.where(rebuilt_end.between(start, EXTRACTION_DATE))
df.loc[flags["start_after_end"], "membership_end"] = rebuilt_end[flags["start_after_end"]]

# Swipe outside the membership: rebuild from the reference date.
reference = df["membership_end"].fillna(EXTRACTION_DATE)
flags["swipe_outside"] = (swipe < start) | (swipe > EXTRACTION_DATE)
rebuilt_swipe = reference - pd.to_timedelta(df["days_since_last_visit"], unit="D")
rebuilt_swipe = rebuilt_swipe.where(rebuilt_swipe.between(start, EXTRACTION_DATE))
df.loc[flags["swipe_outside"], "last_badge_swipe"] = rebuilt_swipe[flags["swipe_outside"]]

# Last-period visits above lifetime visits: total_visits agrees with weekly rate x tenure,
# so the last-period count is the column at fault.
flags["last_period_above_total"] = df["actual_visits_last_period"] > df["total_visits"]
df.loc[flags["last_period_above_total"], "actual_visits_last_period"] = df["total_visits"]

# A swipe after the end date is a soft rule (late check-out), so it is only flagged.
flags["swipe_after_end"] = ~df["is_active"] & (df["last_badge_swipe"] > df["membership_end"])

In [9]:
flags.columns = [f"flag_{c}" for c in flags.columns]
df = df.join(flags)

# How many date rebuilds succeeded (the rest are now missing).
for flag, col in [("flag_start_after_end", "membership_end"), ("flag_swipe_outside", "last_badge_swipe")]:
    rows = df[flag]
    print(f"{flag}: {rows.sum()} rows, {df.loc[rows, col].notna().sum()} rebuilt, {df.loc[rows, col].isna().sum()} set to missing")

flag_report = pd.DataFrame({
    "rows": flags.sum(),
    "share_%": flags.mean() * 100,
}).sort_values("rows", ascending=False)
flag_report

flag_start_after_end: 152 rows, 136 rebuilt, 16 set to missing
flag_swipe_outside: 128 rows, 79 rebuilt, 49 set to missing


,rows,share_%
flag_days_code_999,1298,8.40
flag_last_period_above_total,1127,7.29
flag_colleagues,565,3.66
flag_negative_value,159,1.10
flag_start_after_end,152,0.98
flag_age,143,0.93
flag_swipe_outside,128,0.83
flag_plan_days,97,0.63
flag_swipe_after_end,58,0.38
flag_negative_addon,20,0.13


## 6. Business features

The features follow the three perspectives in the project guidelines (value, behaviour and lifecycle). Notebook 02 showed that cumulative totals mostly grow with tenure, so we prefer rates per month or per week.

| Feature | Definition |
|---|---|
| tenure_days, tenure_months | Start to end date, or to the extraction date for active members |
| start_cohort| Quarter in which the membership started |
| monthly_value | Lifetime value / tenure in months (minimum one month) |
| value_band | Low, mid or high, by terciles of monthly_value |
| utilization_rate | Last-period visits / entitled visits |
| visits_per_plan_day | Average weekly visits / plan days per week |
| recency_days, recency_band | Days since last visit, grouped into bands |
| addon_count | Number of add-ons the member has used (0 to 9) |
| member_state | Churned, new, dormant or active (rules below) |
| flag_extreme_value | Monthly value above ten times the plan median (flag only) |
| quality_issues | Number of cleaning flags on the row |

Member state rules, applied in this order:
1. **Churned:** the membership has an end date.
2. **New:** active, with tenure of 90 days or less.
3. **Dormant:** active, with more than 60 days since the last visit.
4. **Active:** every other active member. Active members with no valid visit date are counted here.

In [10]:
effective_end = df["membership_end"].fillna(EXTRACTION_DATE)
df["tenure_days"] = (effective_end - df["membership_start"]).dt.days.clip(lower=1)
df["tenure_months"] = df["tenure_days"] / 30.44
df["start_cohort"] = df["membership_start"].dt.to_period("Q").astype("string")

df["monthly_value"] = df["lifetime_value"] / df["tenure_months"].clip(lower=1)
df["value_band"] = pd.qcut(df["monthly_value"], 3, labels=["Low", "Mid", "High"]).astype("string")

df["utilization_rate"] = df["actual_visits_last_period"] / df["entitled_visits_last_period"]
df["visits_per_plan_day"] = df["avg_weekly_visits"] / df["plan_days_per_week"]

df["recency_days"] = df["days_since_last_visit"].clip(lower=0)
df["recency_band"] = pd.cut(
    df["recency_days"], [-1, 7, 30, 60, 180, np.inf],
    labels=["0-7 days", "8-30 days", "31-60 days", "61-180 days", "180+ days"],
).astype("string")

ADDONS = ["uses_meeting_rooms", "uses_event_space", "uses_guest_passes", "uses_print",
          "uses_mail_handling", "uses_phone_booths", "uses_day_passes", "uses_storage", "uses_bike_locker"]
df[ADDONS] = df[ADDONS].astype(bool)
df["addon_count"] = df[ADDONS].sum(axis=1)

df["member_state"] = np.select(
    [~df["is_active"], df["tenure_days"] <= NEW_MEMBER_DAYS, df["recency_days"] > DORMANT_DAYS],
    ["Churned", "New", "Dormant"],
    default="Active",
)

# Flag only: monthly values more than ten times the plan median (lifetime values of up to EUR 3.4 million).
# These are kept unchanged; the dashboard can exclude them with the other flagged rows.
plan_typical = df.groupby("plan_tier")["monthly_value"].transform("median")
df["flag_extreme_value"] = df["monthly_value"] > 10 * plan_typical

flag_cols = [c for c in df.columns if c.startswith("flag_")]
df["quality_issues"] = df[flag_cols].sum(axis=1)

df["member_state"].value_counts()

member_state
Churned    11496
New         1586
Dormant     1305
Active      1063
Name: count, dtype: int64

In [11]:
# Value band cut-offs, reported so they can be quoted in the slides.
cuts = df["monthly_value"].quantile([1 / 3, 2 / 3])
print(f"Value bands (EUR per month): Low < {cuts.iloc[0]:,.0f} <= Mid < {cuts.iloc[1]:,.0f} <= High")

df.groupby("plan_tier", observed=True)[["monthly_value", "utilization_rate", "visits_per_plan_day", "addon_count"]].median()

Value bands (EUR per month): Low < 148 <= Mid < 244 <= High


,monthly_value,utilization_rate,visits_per_plan_day,addon_count
plan_tier,,,,
dedicated_desk,393.02,0.74,0.77,3.00
flex,195.23,0.57,0.63,1.00
full_time,304.74,0.32,0.32,2.00
hot_desk,128.45,0.46,0.48,1.00
private_office,697.20,0.15,0.15,3.00


## 7. Checks before saving

These assertions stop the notebook if a rule did not work as intended.

In [12]:
assert df["member_id"].is_unique
assert df["age"].dropna().between(16, 100).all()
assert df["plan_days_per_week"].dropna().isin(PLAN_DAYS.values()).all()
both = df[["actual_visits_last_period", "total_visits"]].dropna()
assert (both["actual_visits_last_period"] <= both["total_visits"]).all()
assert (df.loc[df["is_active"], "membership_end"].isna()).all()
assert (df["membership_end"].dropna() >= df.loc[df["membership_end"].notna(), "membership_start"]).all()
for col in ["total_visits", "n_renewals", "lifetime_value", *ADDON_QTY]:
    assert (df[col].dropna() >= 0).all(), col
print("All checks passed.")

All checks passed.


## 8. Save for the dashboard

We keep the analysis columns and the flags, with simple types (strings, floats, booleans and dates) so the parquet file opens with any recent pandas.

In [13]:
KEEP = [
    "member_id", "age", "gender", "industry", "country", "is_team_account", "n_colleagues_on_account",
    "plan_tier", "plan_days_per_week", "membership_start", "membership_end", "last_badge_swipe",
    "days_since_last_visit", "total_visits", "avg_weekly_visits", "entitled_visits_last_period",
    "actual_visits_last_period", "meeting_room_hours", "event_space_hours", "guest_passes_issued",
    *ADDONS, "n_renewals", "lifetime_value",
    "is_active", "tenure_days", "tenure_months", "start_cohort", "monthly_value", "value_band",
    "utilization_rate", "visits_per_plan_day", "recency_days", "recency_band", "addon_count",
    "member_state", "quality_issues", *flag_cols,
]
dash_df = df[KEEP].reset_index(drop=True)

OUT_FILE.parent.mkdir(parents=True, exist_ok=True)
dash_df.to_parquet(OUT_FILE, index=False)
print(f"Saved {OUT_FILE.relative_to(ROOT)}: {dash_df.shape[0]:,} rows x {dash_df.shape[1]} columns")

Saved data\processed\acdc_dash.parquet: 15,450 rows x 56 columns


In [14]:
# A short data dictionary for the engineered and flag columns, written next to the other docs.
DESCRIPTIONS = {
    "is_active": "True when the raw end date was the 1970-01-01 placeholder",
    "tenure_days": "Days from start to end date, or to 2024-01-01 for active members",
    "tenure_months": "tenure_days / 30.44",
    "start_cohort": "Quarter of membership_start, e.g. 2023Q4",
    "monthly_value": "lifetime_value / tenure_months, with tenure floored at one month",
    "value_band": "Low / Mid / High by terciles of monthly_value",
    "utilization_rate": "actual_visits_last_period / entitled_visits_last_period",
    "visits_per_plan_day": "avg_weekly_visits / plan_days_per_week",
    "recency_days": "days_since_last_visit, negatives set to 0",
    "recency_band": "recency_days grouped into 0-7, 8-30, 31-60, 61-180 and 180+ days",
    "addon_count": "Number of the nine add-ons the member has used",
    "member_state": f"Churned (has end date), New (active, tenure <= {NEW_MEMBER_DAYS} days), "
                    f"Dormant (active, > {DORMANT_DAYS} days since last visit), Active (other active members)",
    "quality_issues": "Number of flag_* columns that are True",
}
DESCRIPTIONS.update({c: "True when this cleaning rule changed or questioned the row" for c in flag_cols})

lines = ["# Dashboard data dictionary", "",
         "Engineered and flag columns in `data/processed/acdc_dash.parquet`, created by `notebook/03_dash_features.ipynb`.",
         "Original columns keep the meaning given in the project guidelines.", "",
         "| Column | Definition |", "|---|---|"]
lines += [f"| `{k}` | {v} |" for k, v in DESCRIPTIONS.items()]
DICT_FILE.parent.mkdir(parents=True, exist_ok=True)
DICT_FILE.write_text("\n".join(lines) + "\n", encoding="utf-8")
print(f"Saved {DICT_FILE.relative_to(ROOT)}")

Saved docs\dash_data_dictionary.md
